# Progress 3 — Paper Baseline Complete Transformer Runs (Colab)

Notebook ini menjalankan baseline transformer IdSarcasm secara lebih lengkap sesuai recipe baseline paper: **6 model × 2 dataset = 12 run**.

Target baseline paper:
- IndoBERT Base IndoNLU
- IndoBERT Large IndoNLU
- IndoBERT Base IndoLEM
- mBERT
- XLM-R Base
- XLM-R Large

Dataset:
- Twitter Indonesia Sarcastic
- Reddit Indonesia Sarcastic

Catatan: setting default runner tetap paper-faithful. Jika model large OOM di Colab, gunakan fallback terpisah dan catat sebagai non-strict paper batch fallback.


## 1. Cek GPU dan versi library

Pastikan runtime Colab memakai GPU. Cell ini juga menyimpan versi library ke log agar perbedaan environment dengan paper bisa dilaporkan.


In [ ]:
!nvidia-smi

import sys, pathlib, subprocess
try:
    import torch, transformers, datasets, accelerate
    print('python', sys.version)
    print('torch', torch.__version__)
    print('transformers', transformers.__version__)
    print('datasets', datasets.__version__)
    print('accelerate', accelerate.__version__)
    print('cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Version check will work after dependencies are installed:', exc)


## 2. Clone / masuk repo dan update


In [ ]:
import pathlib
repo = pathlib.Path.cwd()
if repo.name != 'idsarcasm-reproduction':
    if not pathlib.Path('idsarcasm-reproduction').exists():
        !git clone https://github.com/feb027/idsarcasm-reproduction.git
    %cd idsarcasm-reproduction
else:
    print(f'Already in repo: {repo}')
!git pull


## 3. Install dependency

Opsi default: pakai `requirements.txt` repo. Jika ingin eksperimen environment pinned, jalankan cell pinned di bawah setelah runtime baru lalu restart runtime.


In [ ]:
!pip install -q -r requirements.txt


### 3a. Opsional — pinned environment untuk reproducibility

Jalankan hanya jika ingin runtime baru yang lebih stabil. Setelah cell ini selesai, restart runtime, lalu ulang dari Step 1/2.


In [ ]:
# Optional only. Uncomment if needed, then restart runtime.
# !pip install -q "transformers==4.44.2" "accelerate==0.33.0" "datasets>=2.14" "evaluate>=0.4"


## 4. Cache dataset


In [ ]:
!python scripts/download_data.py


## 5. Smoke test cepat

Smoke test memakai sample limit, jadi otomatis menulis ke `results/tables/transformer_smoke.csv`, bukan tabel final. Log disimpan untuk bukti.


In [ ]:
!mkdir -p results/logs
!python scripts/run_transformer_baseline.py   --dataset twitter   --model indobert-base   --epochs 1   --batch-size 8   --eval-batch-size 16   --max-train-samples 64   --max-eval-samples 32   --max-predict-samples 32   --output-dir results/transformer/smoke-twitter-indobert-base   --model-output-dir models/transformer/smoke-twitter-indobert-base   2>&1 | tee results/logs/progress-3-colab-training-smoke-test.log


## 6. Paper baseline complete — daftar 12 run

Cell ini membuat helper untuk menjalankan run satu per satu sambil menyimpan log. Default-nya memakai setting paper:

- epochs 100
- train/eval batch 32/64
- learning rate 1e-5
- scheduler cosine
- weight decay 0.03
- label smoothing 0.0
- max length 128
- early stopping threshold 0.01
- seed 42
- pad-to-max-length, shuffle train, fp16


In [ ]:
from pathlib import Path
import subprocess, sys, shlex, csv, json, os

Path('results/logs').mkdir(parents=True, exist_ok=True)

PAPER_BASELINE_RUNS = [
    ('twitter', 'indobert-base'),
    ('twitter', 'indobert-large'),
    ('twitter', 'indobert-indolem-base'),
    ('twitter', 'mbert-base'),
    ('twitter', 'xlmr-base'),
    ('twitter', 'xlmr-large'),
    ('reddit', 'indobert-base'),
    ('reddit', 'indobert-large'),
    ('reddit', 'indobert-indolem-base'),
    ('reddit', 'mbert-base'),
    ('reddit', 'xlmr-base'),
    ('reddit', 'xlmr-large'),
]

QUIET_PROGRESS = True  # keeps Colab logs compact; set False if you want tqdm progress bars

PAPER_TARGET_F1 = {
    ('twitter', 'indobert-base'): 0.7273,
    ('twitter', 'indobert-large'): 0.7160,
    ('twitter', 'indobert-indolem-base'): 0.6462,
    ('twitter', 'mbert-base'): 0.6467,
    ('twitter', 'xlmr-base'): 0.7386,
    ('twitter', 'xlmr-large'): 0.7692,
    ('reddit', 'indobert-base'): 0.6100,
    ('reddit', 'indobert-large'): 0.6184,
    ('reddit', 'indobert-indolem-base'): 0.5671,
    ('reddit', 'mbert-base'): 0.5338,
    ('reddit', 'xlmr-base'): 0.5690,
    ('reddit', 'xlmr-large'): 0.6274,
}

def result_path(dataset, model):
    return Path(f'results/transformer/{dataset}-{model}/result_row.json')

def log_path(dataset, model, suffix=''):
    suffix = f'-{suffix}' if suffix else ''
    return Path(f'results/logs/progress-3-{dataset}-{model}{suffix}.log')

def build_cmd(dataset, model, *, batch_size=32, eval_batch_size=64, gradient_accumulation_steps=1,
              gradient_checkpointing=False, auto_find_batch_size=False, output_suffix=None, quiet_progress=QUIET_PROGRESS):
    run_name = output_suffix or f'{dataset}-{model}'
    cmd = [
        sys.executable, 'scripts/run_transformer_baseline.py',
        '--dataset', dataset,
        '--model', model,
        '--epochs', '100',
        '--batch-size', str(batch_size),
        '--eval-batch-size', str(eval_batch_size),
        '--learning-rate', '1e-5',
        '--lr-scheduler-type', 'cosine',
        '--weight-decay', '0.03',
        '--label-smoothing-factor', '0.0',
        '--max-length', '128',
        '--early-stopping-threshold', '0.01',
        '--seed', '42',
        '--pad-to-max-length',
        '--shuffle-train-dataset',
        '--fp16',
        '--gradient-accumulation-steps', str(gradient_accumulation_steps),
        '--output-dir', f'results/transformer/{run_name}',
        '--model-output-dir', f'models/transformer/{run_name}',
    ]
    if quiet_progress:
        cmd.append('--disable-tqdm')
    if gradient_checkpointing:
        cmd.append('--gradient-checkpointing')
    if auto_find_batch_size:
        cmd.append('--auto-find-batch-size')
    return cmd

def run_baseline(dataset, model, *, skip_if_done=True, fallback=False):
    """Run one baseline and stream+save Colab log."""
    if not fallback:
        out_name = f'{dataset}-{model}'
        lp = log_path(dataset, model)
        cmd = build_cmd(dataset, model)
    else:
        out_name = f'{dataset}-{model}-colab-fallback'
        lp = log_path(dataset, model, 'colab-fallback')
        cmd = build_cmd(
            dataset, model,
            batch_size=4,
            eval_batch_size=8,
            gradient_accumulation_steps=8,
            gradient_checkpointing=True,
            output_suffix=out_name,
        )

    rp = Path(f'results/transformer/{out_name}/result_row.json')
    if skip_if_done and rp.exists():
        print(f'SKIP existing result: {rp}')
        return 0

    print('Running one-line command:', ' '.join(shlex.quote(x) for x in cmd))
    print('Log:', lp)
    env = os.environ.copy()
    if QUIET_PROGRESS:
        env.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
        env.setdefault('HF_DATASETS_DISABLE_PROGRESS_BARS', '1')
        env.setdefault('TOKENIZERS_PARALLELISM', 'false')
    with lp.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
        rc = proc.wait()
    print('Exit code:', rc)
    if rc != 0:
        print(f'FAILED. Keep log for report: {lp}')
    return rc

def show_progress_table():
    rows = []
    for dataset, model in PAPER_BASELINE_RUNS:
        rp = result_path(dataset, model)
        if rp.exists():
            row = json.loads(rp.read_text())
            f1 = float(row['f1'])
            target = PAPER_TARGET_F1[(dataset, model)]
            rows.append({
                'dataset': dataset,
                'model': model,
                'status': 'done',
                'f1': f1,
                'paper_f1': target,
                'gap': round(f1 - target, 4),
            })
        else:
            rows.append({
                'dataset': dataset,
                'model': model,
                'status': 'missing',
                'f1': None,
                'paper_f1': PAPER_TARGET_F1[(dataset, model)],
                'gap': None,
            })
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)

show_progress_table()


## 7. Jalankan run berikutnya satu per satu

Rekomendasi: jalankan satu per satu, jangan paralel di GPU yang sama. Kalau ada OOM pada model large, biarkan log gagal tersimpan, lalu boleh coba fallback di section berikutnya.


In [ ]:
# Contoh run berikutnya. Uncomment satu per satu sesuai kebutuhan.

# run_baseline('reddit', 'indobert-base')
# run_baseline('reddit', 'xlmr-base')
# run_baseline('twitter', 'mbert-base')
# run_baseline('twitter', 'indobert-indolem-base')
# run_baseline('reddit', 'mbert-base')
# run_baseline('reddit', 'indobert-indolem-base')
# run_baseline('twitter', 'indobert-large')
# run_baseline('reddit', 'indobert-large')
# run_baseline('twitter', 'xlmr-large')
# run_baseline('reddit', 'xlmr-large')


## 8. Opsional — jalankan semua yang belum ada

Hati-hati: cell ini bisa memakan waktu lama. Jika runtime Colab terbatas, lebih baik jalankan manual satu per satu dari section 7.


In [ ]:
# Uncomment kalau benar-benar ingin attempt semua run yang belum ada.
# for dataset, model in PAPER_BASELINE_RUNS:
#     rc = run_baseline(dataset, model, skip_if_done=True)
#     if rc != 0:
#         print('Stopping after failure. Check log, optionally try fallback for large models.')
#         break
# show_progress_table()


## 9. Fallback untuk model large jika OOM

Fallback ini **tidak strict paper-faithful** karena batch fisik turun dari 32 ke 4, tetapi effective batch dibuat mendekati 32 dengan gradient accumulation. Gunakan hanya setelah paper-faithful attempt gagal/OOM, dan catat di laporan.


In [ ]:
# Fallback examples after OOM. Uncomment as needed.

# run_baseline('twitter', 'xlmr-large', fallback=True)
# run_baseline('reddit', 'xlmr-large', fallback=True)
# run_baseline('twitter', 'indobert-large', fallback=True)
# run_baseline('reddit', 'indobert-large', fallback=True)


## 10. Lihat hasil dan gap terhadap paper


In [ ]:
show_progress_table()

import pathlib, json
for path in sorted(pathlib.Path('results/transformer').glob('*/result_row.json')):
    print('
---', path, '---')
    print(path.read_text())

print('
Transformer baseline table:')
try:
    import pandas as pd
    display(pd.read_csv('results/tables/transformer_baselines.csv'))
except Exception as exc:
    print(exc)


## 11. Cek log agar tidak ada token/secret


In [ ]:
!grep -RniE "hf_[A-Za-z0-9]|password|secret|api_key|apikey" results/logs || true


## 12. Commit hasil dari Colab

Jalankan setelah hasil/log dicek. Jika Colab belum punya identity/token GitHub, set `git config` dan remote token seperti instruksi di chat sebelumnya.


In [ ]:
!git status --short

# Uncomment setelah siap commit.
# !git add results/tables/transformer_baselines.csv results/tables/transformer_smoke.csv results/transformer/ results/logs/
# !git commit -m "results: extend Progress 3 paper baseline transformer runs"
# !git push
